# Covariance curves and the estimator ladder (he_estimators, shared with CEU/GBR)

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = min(cfg.params.get("threads", 4), os.cpu_count() or 1)
for v in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "OMP_NUM_THREADS"):   # bigsnpr parallelises itself; one BLAS thread
    os.environ[v] = "1"
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

In [ ]:
import he_estimators as he
STAGE = "curves"
require_ready(cfg, STAGE)
rep, OUT = Report(cfg, STAGE), single_target(cfg, STAGE)
mid, NB = he.load_grid(cfg.path("crossproducts", "bins.txt")), cfg.param("nblocks")

In [ ]:
rows, arrays = [], {}
for uri, item in target_items(cfg, "merged"):
    S, N = he.load_arrays(uri, uri.replace(".full.tsv", ".jk.tsv"), len(mid), NB)
    arrays[tuple(item.values())] = (S, N)
    rows.append(he.jackknife(S, N, mid, NB).assign(**item))
est = pd.concat(rows, ignore_index=True)
if needs_run(OUT, FORCE):
    with record(cfg, STAGE, OUT):
        get_store(OUT).write_text(OUT, est.to_csv(sep="\t", index=False))
show = est[est["estimator"].isin(["h2_Unrel", "h2_Rel", "h2_RelOffsets"]) & (est["classes"] == "noPO")]
rep.table(show[["pheno", "transform", "covset", "estimator", "est", "se"]].round(3), "estimates")

In [ ]:
fig, axes = plt.subplots(1, len(cfg.lists["phenotypes"]), figsize=(13, 4.5))
for ax, ph in zip(np.atleast_1d(axes), cfg.lists["phenotypes"]):
    S, N = arrays[(ph, "invnorm", "base_pcs")]
    x, m, n, se = he.bin_curve(S, N, he.NOPO, mid, NB)
    k = (n > 20) & (x < he.R_HI)
    e = est[(est["pheno"] == ph) & (est["transform"] == "invnorm") & (est["covset"] == "base_pcs")
            & (est["classes"] == "noPO")].set_index("estimator")["est"]
    xs = np.linspace(-0.05, he.R_HI, 50)
    ax.plot(xs, e["h2_Unrel"] * xs, color="tab:blue", lw=1, zorder=1, label="h2_Unrel")
    ax.plot(xs[xs > he.U_HI], e["h2_Rel"] * xs[xs > he.U_HI] + e["Rel.intercept"], color="tab:orange",
            lw=1, zorder=1, label="h2_Rel")
    ax.errorbar(x[k], m[k], yerr=se[k], fmt="o", ms=2, color="0.2", lw=0.6, zorder=2)
    ax.set(title=ph, xlabel="relatedness bin", ylabel="mean y_i y_j"); ax.legend(fontsize=8)
rep.fig(fig, "covariance_curves", "invnorm, base_pcs; bins with <= 20 pairs not shown"); plt.show()

In [ ]:
if cfg.params.get("simulated"):
    truth = json.load(open(cfg.input("truth")))
    for ph in cfg.lists["phenotypes"]:
        r = est[(est["pheno"] == ph) & (est["transform"] == "invnorm") & (est["covset"] == "base_pcs")
                & (est["estimator"] == "h2_Unrel")].iloc[0]
        rep.check(f"{ph}: h2_Unrel recovers the simulated h2 ({truth[ph]['h2']})",
                  abs(r.est - truth[ph]["h2"]) < 3 * r.se, f"{r.est:.3f} ± {r.se:.3f}")
rep.save()